# 05장. 당근마켓 키워드 검색 기반 중고거래 데이터 스크래핑 실습

- **학습 순서**:
  1. 필요 패키지 로드
  2. 크롬 웹드라이버 구동 및 당근마켓 메인 접속
  3. **'중고거래' 탭 먼저 확인 및 클릭** (검색어 날아감 방지)
  4. **검색창에 키워드('맥북 프로') 입력 및 검색 실행**
  5. **검색된 매물 카드 로딩 확인**
  6. 단일 매물 카드의 텍스트와 속성 구조 분석
  7. 카드 텍스트 파싱 함수 작성
  8. '더보기' 버튼 클릭으로 추가 매물 수집
  9. Pandas DataFrame 변환 및 정제
  10. CSV 저장 및 브라우저 종료
  11. (선택) PostgreSQL DB 테이블 저장


## 1. 필요 패키지 로드


In [ ]:
import time
import re
import random
from datetime import datetime
from pathlib import Path

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By #폴더.폴더.파일명 여기서 .은 경로를 의미
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


## 2. 크롬 웹드라이버 구동 및 당근마켓 메인 접속


In [3]:
options = webdriver.ChromeOptions()
options.add_argument('--window-size=1920,1080')
options.add_argument('--disable-blink-features=AutomationControlled')

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

# 당근마켓 메인 페이지 접속
driver.get('https://www.daangn.com')
print('접속 완료:', driver.title)


접속 완료: 당신 근처의 당근


## 3. [순서 1] '중고거래' 탭(메뉴) 먼저 확인하고 클릭
- 검색어를 먼저 치고 탭을 누르면 검색어가 초기화되는 문제를 해결하기 위해, **'중고거래' 탭을 먼저 클릭**하여 중고거래 전용 화면으로 이동합니다.


In [4]:
# 상단 메뉴에서 '중고거래' 탭 탐색 및 클릭
used_tab = wait.until(
    EC.element_to_be_clickable((By.XPATH, "//a[contains(text(), '중고거래')] | //button[contains(text(), '중고거래')]"))
)
used_tab.click()
time.sleep(2.5)
print('중고거래 전용 화면 이동 완료 URL:', driver.current_url)


중고거래 전용 화면 이동 완료 URL: https://www.daangn.com/kr/search/buy-sell/


## 4. [순서 2] 검색창에 '맥북 프로' 입력 및 검색 실행
- 중고거래 전용 화면의 검색창을 찾아 검색어를 입력하고 Enter를 누릅니다.


In [5]:
keyword = '맥북 프로'

# 중고거래 화면의 검색창 input 요소 탐색
search_input = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "input[type='search'], input[type='text'], input[placeholder*='검색']"))
)

# 검색창 비우기 및 키워드 입력 후 엔터
search_input.clear()
search_input.send_keys(keyword)
search_input.send_keys(Keys.ENTER)
print(f'"{keyword}" 검색 실행 완료!')


"맥북 프로" 검색 실행 완료!


## 5. [순서 3] 검색된 매물 카드 로딩 확인
- 검색 결과 페이지에서 중고 매물 카드(`a[href*="/kr/buy-sell/"]`)들이 정상적으로 로딩되었는지 확인합니다.


In [18]:
time.sleep(3)  # 검색 결과 렌더링 대기

# 검색된 매물 카드 목록 가져오기
article_cards = wait.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'a[href*="/kr/buy-sell/"]'))
)
print(f'"{keyword}" 검색 결과 매물 카드 수: {len(article_cards)}개')


"맥북 프로" 검색 결과 매물 카드 수: 32개


## 6. 단일 매물 카드의 텍스트와 속성 구조 뜯어보기


In [19]:
sample_card = article_cards[0]

print('=== [카드 전체 텍스트] ===')
print(sample_card.text)

print('\n=== [상세 링크 URL] ===')
print(sample_card.get_attribute('href'))


=== [카드 전체 텍스트] ===
맥북 프로 14인치 M1 Pro 16GB 1TB
1,380,000원
숭인제2동
 · 
1시간 전

=== [상세 링크 URL] ===
https://www.daangn.com/kr/buy-sell/%EB%A7%A5%EB%B6%81-%ED%94%84%EB%A1%9C-14%EC%9D%B8%EC%B9%98-m1-pro-16gb-1tb-3tmreo4yyr1u/


## 7. 카드 텍스트 파싱 함수 작성
- 제목, 가격(숫자 정제), 동네, 시간 정보를 분리 추출합니다.


In [ ]:
# 단일 매물 카드 웹 요소(element)와 검색어(keyword_val)를 인자로 받아 데이터를 추출하는 함수 정의
def parse_daangn_item(element, keyword_val=''):
    """단일 매물 카드 요소에서 중고거래 정보를 추출하는 함수"""
    # 카드 요소(<a> 태그) 내부에 적힌 모든 글자들을 한 번에 가져오고, 문자열 앞뒤의 불필요한 공백 제거
    raw_text = element.text.strip()
    # <a> 태그의 href 속성값(클릭 시 이동할 상세 페이지 URL 주소) 가져오기
    link_url = element.get_attribute('href')
    # 전체 텍스트를 줄바꿈('\n') 기준으로 자른 뒤, 빈 줄이나 'thumbnail', '바로구매' 같은 부가 텍스트는 제외하고 모음
    lines = [line.strip() for line in raw_text.split('\n') if line.strip() and line.strip() not in ['thumbnail', '바로구매']]
    # 1. 쪼갠 줄 중에서 가장 첫 번째 줄(0번 인덱스)을 상품 제목으로 지정 (내용이 없으면 빈 문자열)
    title = lines[0] if len(lines) > 0 else ''
    # 2. 가격 정보를 담을 변수를 생성하고 기본값 None(값 없음)으로 초기화
    price_val = None
    # 정규표현식: 텍스트 안에서 '숫자 및 쉼표 + 원' 패턴(예: "1,380,000원")을 검색하여 매칭 객체 생성
    price_match = re.search(r'([\d,]+)원', raw_text)
    # 정규식 패턴과 일치하는 가격 문자열을 찾았을 경우
    if price_match:
        # re.sub로 쉼표(,) 등 숫자가 아닌 모든 문자([^0-9])를 제거하여 순수 숫자로 만든 뒤 정수(int)로 변환
        price_val = int(re.sub(r'[^0-9]', '', price_match.group(1)))
    # 가격 표시는 없지만 '나눔'이라는 단어가 텍스트에 포함되어 있는 경우 (무료 나눔 물품)
    elif '나눔' in raw_text:
        # 무료 나눔이므로 가격을 0원으로 지정
        price_val = 0
    # 3. 거래 지역(동네) 이름을 저장할 변수 초기화
    region = ''
    # 등록 또는 끌올 시간 정보를 저장할 변수 초기화
    time_text = ''
    # 제목(0번 줄)을 제외하고 두 번째 줄(1번 줄)부터 끝까지 차례대로 확인 (슬라이싱 [1:])
    for line in lines[1:]:
        # 가격이 적힌 줄은 지역/시간 정보가 아니므로 건너뛰기
        if '원' not in line:
            # 당근마켓의 구분 기호인 가운뎃점('·')이 포함된 경우 (예: '숭인제2동 · 43분 전')
            if '·' in line:
                # 가운뎃점을 기준으로 동네와 시간 두 조각으로 분리 (['숭인제2동 ', ' 43분 전'])
                parts = line.split('·')
                # 앞부분을 동네 이름으로 저장 (앞뒤 공백 제거)
                region = parts[0].strip()
                # 뒷부분이 존재하면 등록 시간으로 저장 (없으면 빈 문자열)
                time_text = parts[1].strip() if len(parts) > 1 else ''
                # 동네와 시간 정보를 모두 찾았으므로 더 이상 줄을 검사하지 않고 반복문 탈출
                break
            # 가운뎃점이 없더라도 행정구역 단위('동','구','읍','면')나 시간 문구('전')가 들어있는 경우
            elif any(s in line for s in ['동', '구', '읍', '면', '전']):
                # 해당 줄 전체를 지역 정보로 저장
                region = line
                # 정보를 찾았으므로 반복문 탈출
                break
    # 정제된 6가지 핵심 항목을 딕셔너리 형태로 묶어 최종 반환
    return {
        'keyword': keyword_val,     # 검색에 사용된 검색어 (예: '맥북 프로')
        'title': title,             # 상품 제목
        'price': price_val,         # 판매 가격 (정수형 숫자)
        'region': region,           # 거래 지역 (동네명)
        'time_info': time_info if 'time_info' in locals() else time_text, # 등록/끌올 시간
        'article_url': link_url,    # 상품 상세 페이지 URL 링크
    }
# 위에서 찾은 첫 번째 샘플 카드(sample_card)와 검색어(keyword)를 함수에 넣어 파싱 테스트 실행
parse_daangn_item(sample_card, keyword)

{'keyword': '맥북 프로',
 'title': '맥북 프로 14인치 M1 Pro 16GB 1TB',
 'price': 1380000,
 'region': '숭인제2동',
 'time_info': '',
 'article_url': 'https://www.daangn.com/kr/buy-sell/%EB%A7%A5%EB%B6%81-%ED%94%84%EB%A1%9C-14%EC%9D%B8%EC%B9%98-m1-pro-16gb-1tb-3tmreo4yyr1u/'}

## 8. '더보기' 버튼을 클릭하여 추가 매물 수집 (확장 로딩)


In [23]:
target_posts = 200  # 목표 수집 개수
max_clicks = 6     # 더보기 최대 클릭 횟수

for click_no in range(max_clicks):
    current_elements = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/kr/buy-sell/"]')
    print(f'현재 확보된 매물: {len(current_elements)}개 / 목표: {target_posts}개')

    if len(current_elements) >= target_posts:
        print('목표 수집 개수를 충족했습니다!')
        break

    try:
        # 더보기 버튼 탐색
        more_btn = driver.find_element(By.XPATH, "//button[contains(text(), '더보기')]")
        driver.execute_script("arguments[0].scrollIntoView(true);", more_btn)
        time.sleep(1)
        more_btn.click()
        print(f'[{click_no + 1}회] 더보기 버튼 클릭 완료')
        time.sleep(random.uniform(2.0, 3.5))
    except Exception as click_err:
        print('더보기 버튼 탐색 실패 (스크롤 다운 대체 시도):', click_err)
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
        time.sleep(2.5)
        break

# 최종 목표 개수만큼 요소 슬라이싱
final_cards = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/kr/buy-sell/"]')[:target_posts]
print(f'최종 파싱 대상 매물 수: {len(final_cards)}개')


현재 확보된 매물: 32개 / 목표: 200개
[1회] 더보기 버튼 클릭 완료
현재 확보된 매물: 32개 / 목표: 200개
[2회] 더보기 버튼 클릭 완료
현재 확보된 매물: 274개 / 목표: 200개
목표 수집 개수를 충족했습니다!
최종 파싱 대상 매물 수: 200개


## 9. 수집 데이터 Pandas DataFrame 변환 및 정제


In [24]:
rows = [parse_daangn_item(card, keyword) for card in final_cards]

df_daangn = pd.DataFrame(rows)
print('정제 전 데이터 건수:', len(df_daangn))

# 1. 중복 매물 제거 (article_url 기준)
df_daangn = df_daangn.drop_duplicates(subset=['article_url']).reset_index(drop=True)
# 2. 인덱스 열 생성
df_daangn['index'] = df_daangn.index

print('정제 후 데이터 건수:', len(df_daangn))
df_daangn.head()


정제 전 데이터 건수: 200
정제 후 데이터 건수: 200


,keyword,title,price,region,time_info,article_url,index
0,맥북 프로,맥북 프로 14인치 M1 Pro 16GB 1TB,1380000,숭인제2동,,https://www.daangn.com/kr/buy-sell/%EB%A7%A5%E...,0
1,맥북 프로,맥북프로 13인치 M1 8GB / 256GB,640000,청라1동,,https://www.daangn.com/kr/buy-sell/%EB%A7%A5%E...,1
2,맥북 프로,[삽니다] 맥북프로 16인치 M4 Pro 24GB 512GB,2600000,,,https://www.daangn.com/kr/buy-sell/%EC%82%BD%E...,2
3,맥북 프로,맥북 프로 2020 13인치 스페이스 그레이,490000,공덕동,,https://www.daangn.com/kr/buy-sell/%EB%A7%A5%E...,3
4,맥북 프로,맥북 프로 13인치 스페이스 그레이,200000,청평면,,https://www.daangn.com/kr/buy-sell/%EB%A7%A5%E...,4


## 10. CSV 파일 저장 및 브라우저 종료


In [25]:
data_path = Path('../data')
data_path.mkdir(parents=True, exist_ok=True)

now_ts = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_filename = data_path / f'daangn_{keyword}_{now_ts}.csv'

df_daangn.to_csv(csv_filename, index=False, encoding='utf-8-sig')
print('CSV 저장 완료:', csv_filename.resolve())

driver.quit()
print('브라우저 정상 종료')


CSV 저장 완료: D:\SourceBank\ai-datapipeline-2026\data\daangn_맥북 프로_20261007_155527.csv
브라우저 정상 종료


## 11. (선택 실습) PostgreSQL DB 테이블 저장
- chapt05에서 구축한 로컬 PostgreSQL DB에 저장합니다.


In [26]:
from sqlalchemy import create_engine, URL, BigInteger, Text, Integer

try:
    db_url = URL.create(
        'postgresql+psycopg',
        username='postgres',
        password='123456',
        host='127.0.0.1',
        port=5432,
        database='postgres'
    )
    engine = create_engine(db_url)
    table_name = f"daangn_{datetime.now().strftime('%y%m%d')}"

    df_daangn.to_sql(
        name=table_name,
        con=engine,
        if_exists='replace',
        index=False,
        dtype={
            'index': BigInteger(),
            'keyword': Text(),
            'title': Text(),
            'price': Integer(),
            'region': Text(),
            'time_info': Text(),
            'article_url': Text(),
        }
    )
    print(f'PostgreSQL 테이블 [{table_name}] 저장 완료 ({len(df_daangn)}건)')
    engine.dispose()
except Exception as err:
    print('DB 저장 생략 또는 오류:', err)


PostgreSQL 테이블 [daangn_261007] 저장 완료 (200건)
